# ONNX Export and ONNX Runtime

ONNX is the bridge between training code and portable inference. In this notebook you will export a PyTorch model to ONNX, validate the exported graph, run inference with ONNX Runtime, compare outputs, benchmark latency, and prepare the exported model for Triton.

## What you will learn

By the end, you should be able to explain and use:

- why ONNX matters for model serving,
- static vs dynamic input shapes,
- ONNX opsets, input names, and output names,
- PyTorch to ONNX export,
- `onnx.checker` validation,
- ONNX Runtime inference,
- output parity checks against PyTorch,
- latency comparison between PyTorch and ONNX Runtime,
- and how ONNX fits into Triton.

## 1. Why ONNX exists

Training usually happens in a framework such as PyTorch. Production serving often wants a stable artifact that can run outside the training codebase. ONNX gives you a portable model graph that can be executed by runtimes such as ONNX Runtime, TensorRT, OpenVINO, or Triton's ONNX backend.

For Week 3-4, learn ONNX as a practical deployment artifact. You do not need to memorize the whole ONNX specification. Focus on export, validation, runtime inference, shape handling, and debugging export failures.

## 2. Core concepts

| Concept | Meaning | Why it matters |
|---|---|---|
| Graph | The exported computation DAG | Serving runtimes execute this graph |
| Node | One operation in the graph | Unsupported nodes can block deployment |
| Initializer | Stored model weight | These replace PyTorch parameters |
| Opset | ONNX operator version set | Newer opsets support newer behavior |
| Input name | Runtime input key | Client code must use the exact name |
| Output name | Runtime output key | Serving configs often reference it |
| Dynamic axes | Dimensions allowed to vary | Needed for variable batch sizes |
| Provider | ONNX Runtime execution backend | CPU, CUDA, TensorRT, and others |

## 3. Setup

The notebook is safe to open even if `onnx` or `onnxruntime` are not installed. Export and runtime cells catch missing-package errors and explain what to install.

Useful installs:

```bash
pip install onnx onnxruntime
```

For GPU execution with ONNX Runtime, install the CUDA-compatible ONNX Runtime GPU package for your environment.

In [ ]:
import importlib.util
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def has_package(name):
    return importlib.util.find_spec(name) is not None

print(f"device: {device}")
print(f"onnx installed: {has_package('onnx')}")
print(f"onnxruntime installed: {has_package('onnxruntime')}")

## 4. Define a small model

We will use a compact tabular classifier because it exports quickly and makes serving mechanics easy to see. The same export ideas apply to larger PyTorch models, though transformers can introduce extra export constraints.

In [ ]:
class TinyFraudScorer(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, features):
        return self.net(features)

model = TinyFraudScorer().to(device).eval()
sample_batch = torch.randn(8, 64, dtype=torch.float32)
sample_batch_device = sample_batch.to(device)

with torch.inference_mode():
    logits = model(sample_batch_device)

print(logits.shape)
print(logits[:2])

## 5. Export PyTorch to ONNX

Important export choices:

- `input_names`: the names clients use when sending tensors.
- `output_names`: the names clients or serving configs use for outputs.
- `dynamic_axes`: dimensions that can change at runtime.
- `opset_version`: ONNX operator version set.

For serving, dynamic batch size is usually the first dynamic axis you want.

In [ ]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)
onnx_path = artifacts_dir / "tiny_fraud_scorer.onnx"

export_kwargs = {
    "input_names": ["features"],
    "output_names": ["logits"],
    "dynamic_axes": {
        "features": {0: "batch"},
        "logits": {0: "batch"},
    },
    "opset_version": 17,
    "do_constant_folding": True,
}

try:
    torch.onnx.export(model, sample_batch_device, onnx_path, **export_kwargs)
    print(f"wrote: {onnx_path.resolve()}")
    print(f"size_mb: {onnx_path.stat().st_size / 1_000_000:.3f}")
except Exception as error:
    print("ONNX export failed. Install onnx first, then rerun this cell.")
    print(f"Reason: {error}")

## 6. Validate and inspect the ONNX graph

Export success is not enough. Always load the graph and run `onnx.checker.check_model`. Then inspect names and shapes because those names become part of your serving contract.

In [ ]:
try:
    import onnx

    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print("onnx.checker: ok")
    print(f"ir_version: {onnx_model.ir_version}")
    print(f"opsets: {[opset.version for opset in onnx_model.opset_import]}")
    print(f"inputs: {[value.name for value in onnx_model.graph.input]}")
    print(f"outputs: {[value.name for value in onnx_model.graph.output]}")
    print(f"nodes: {len(onnx_model.graph.node)}")
    print(f"first_ops: {[node.op_type for node in onnx_model.graph.node[:10]]}")
except Exception as error:
    print("ONNX validation skipped. Export the model and install onnx first.")
    print(f"Reason: {error}")

## 7. Run ONNX Runtime inference

ONNX Runtime executes the exported graph. The key serving pattern is:

1. create an inference session,
2. read the required input name,
3. send NumPy arrays with correct dtype and shape,
4. receive NumPy arrays as outputs.

In [ ]:
def create_ort_session(path):
    import onnxruntime as ort

    available = ort.get_available_providers()
    preferred = []
    if "CUDAExecutionProvider" in available:
        preferred.append("CUDAExecutionProvider")
    preferred.append("CPUExecutionProvider")
    return ort.InferenceSession(str(path), providers=preferred)

try:
    ort_session = create_ort_session(onnx_path)
    print(f"providers: {ort_session.get_providers()}")
    print(f"inputs: {[(item.name, item.shape, item.type) for item in ort_session.get_inputs()]}")
    print(f"outputs: {[(item.name, item.shape, item.type) for item in ort_session.get_outputs()]}")
except Exception as error:
    ort_session = None
    print("ONNX Runtime session skipped. Install onnxruntime and export the model first.")
    print(f"Reason: {error}")

In [ ]:
try:
    input_name = ort_session.get_inputs()[0].name
    output_name = ort_session.get_outputs()[0].name
    batch_np = sample_batch.numpy().astype(np.float32)

    ort_logits = ort_session.run([output_name], {input_name: batch_np})[0]
    with torch.inference_mode():
        torch_logits = model(sample_batch_device).detach().cpu().numpy()

    max_abs_diff = np.max(np.abs(torch_logits - ort_logits))
    print(f"max_abs_diff: {max_abs_diff:.8f}")
    print(ort_logits[:2])
except Exception as error:
    print("Parity check skipped. Create an ONNX Runtime session first.")
    print(f"Reason: {error}")

## 8. Benchmark PyTorch vs ONNX Runtime

Benchmarking must include warmup and percentiles. A single average hides tail latency. This benchmark is intentionally small, but the pattern is the same for real services.

In [ ]:
def sync_if_needed():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def percentile(values, q):
    values = sorted(values)
    if not values:
        return float("nan")
    index = min(len(values) - 1, max(0, round((q / 100) * (len(values) - 1))))
    return values[index]

def summarize_latencies(name, latencies_ms, batch_size):
    return {
        "runtime": name,
        "batch_size": batch_size,
        "mean_ms": float(np.mean(latencies_ms)),
        "p50_ms": percentile(latencies_ms, 50),
        "p95_ms": percentile(latencies_ms, 95),
        "p99_ms": percentile(latencies_ms, 99),
        "throughput_items_per_s": batch_size / (float(np.mean(latencies_ms)) / 1000),
    }

@torch.inference_mode()
def benchmark_pytorch(model, batch, warmup=25, repeats=100):
    for _ in range(warmup):
        _ = model(batch)
    sync_if_needed()

    latencies = []
    for _ in range(repeats):
        start = time.perf_counter()
        _ = model(batch)
        sync_if_needed()
        latencies.append((time.perf_counter() - start) * 1000)
    return latencies

def benchmark_onnxruntime(session, batch_np, warmup=25, repeats=100):
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    for _ in range(warmup):
        _ = session.run([output_name], {input_name: batch_np})

    latencies = []
    for _ in range(repeats):
        start = time.perf_counter()
        _ = session.run([output_name], {input_name: batch_np})
        latencies.append((time.perf_counter() - start) * 1000)
    return latencies

In [ ]:
try:
    batch_size = 32
    benchmark_batch = torch.randn(batch_size, 64, dtype=torch.float32)
    benchmark_batch_device = benchmark_batch.to(device)
    benchmark_batch_np = benchmark_batch.numpy().astype(np.float32)

    pytorch_latencies = benchmark_pytorch(model, benchmark_batch_device)
    ort_latencies = benchmark_onnxruntime(ort_session, benchmark_batch_np)

    comparison = pd.DataFrame([
        summarize_latencies("pytorch", pytorch_latencies, batch_size),
        summarize_latencies("onnxruntime", ort_latencies, batch_size),
    ])
    comparison
except Exception as error:
    print("Benchmark skipped. Export the ONNX model and create an ONNX Runtime session first.")
    print(f"Reason: {error}")

In [ ]:
try:
    ax = comparison.set_index("runtime")[["p50_ms", "p95_ms", "p99_ms"]].plot(kind="bar", figsize=(8, 4))
    ax.set_ylabel("latency ms")
    ax.set_title("PyTorch vs ONNX Runtime latency")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=0)
    plt.tight_layout()
except Exception as error:
    print("Plot skipped. Run the benchmark cell first.")
    print(f"Reason: {error}")

## 9. Test dynamic batch sizes

Because we exported the batch axis as dynamic, ONNX Runtime should accept different batch sizes without re-exporting. This is essential for serving systems that batch requests dynamically.

In [ ]:
try:
    rows = []
    for batch_size in [1, 4, 16, 64, 128]:
        batch_np = np.random.randn(batch_size, 64).astype(np.float32)
        output = ort_session.run(None, {input_name: batch_np})[0]
        latencies = benchmark_onnxruntime(ort_session, batch_np, warmup=10, repeats=40)
        row = summarize_latencies("onnxruntime", latencies, batch_size)
        row["output_shape"] = tuple(output.shape)
        rows.append(row)

    dynamic_batch_results = pd.DataFrame(rows)
    dynamic_batch_results
except Exception as error:
    print("Dynamic batch test skipped. Run the ONNX Runtime session cell first.")
    print(f"Reason: {error}")

## 10. Optional: dynamic quantization

Quantization reduces model size and can improve CPU inference latency. Dynamic quantization is one of the easiest places to start for linear-heavy models, but it does not always improve every workload. Always measure.

In [ ]:
quantized_onnx_path = artifacts_dir / "tiny_fraud_scorer_dynamic_quant.onnx"

try:
    from onnxruntime.quantization import QuantType, quantize_dynamic

    quantize_dynamic(str(onnx_path), str(quantized_onnx_path), weight_type=QuantType.QInt8)
    print(f"wrote: {quantized_onnx_path.resolve()}")
    print(f"fp32_size_mb: {onnx_path.stat().st_size / 1_000_000:.3f}")
    print(f"int8_size_mb: {quantized_onnx_path.stat().st_size / 1_000_000:.3f}")
except Exception as error:
    print("Quantization skipped. It requires onnxruntime quantization support and an exported ONNX model.")
    print(f"Reason: {error}")

## 11. Prepare an ONNX model repository for Triton

Triton can serve ONNX models through the ONNX Runtime backend. The repository shape is similar to the PyTorch Triton notebook, but the model file is `model.onnx` and the platform is `onnxruntime_onnx`.

In [ ]:
triton_root = Path("triton_model_repository")
triton_model_name = "tiny_fraud_scorer_onnx"
triton_version_dir = triton_root / triton_model_name / "1"
triton_version_dir.mkdir(parents=True, exist_ok=True)

if onnx_path.exists():
    shutil.copyfile(onnx_path, triton_version_dir / "model.onnx")

config_pbtxt = """name: "tiny_fraud_scorer_onnx"
platform: "onnxruntime_onnx"
max_batch_size: 128

input [
  {
    name: "features"
    data_type: TYPE_FP32
    dims: [64]
  }
]

output [
  {
    name: "logits"
    data_type: TYPE_FP32
    dims: [2]
  }
]

dynamic_batching {
  preferred_batch_size: [8, 16, 32, 64]
  max_queue_delay_microseconds: 1000
}
"""

config_path = triton_root / triton_model_name / "config.pbtxt"
config_path.write_text(config_pbtxt)
print(f"repository: {triton_root.resolve()}")
print(config_path.read_text())

In [ ]:
triton_docker_command = """
docker run --rm \
  -p 8000:8000 -p 8001:8001 -p 8002:8002 \
  -v $PWD/triton_model_repository:/models \
  nvcr.io/nvidia/tritonserver:24.12-py3 \
  tritonserver --model-repository=/models
""".strip()

print(triton_docker_command)

## 12. Common ONNX export failures

| Symptom | Likely cause | Fix |
|---|---|---|
| Export fails with unsupported operator | PyTorch op has no ONNX equivalent for the opset | Try a newer opset, rewrite the layer, or use a different export path |
| Runtime says input name missing | Client used the wrong input key | Inspect `session.get_inputs()` and use the exact name |
| Batch size cannot change | Batch axis was exported static | Re-export with `dynamic_axes` for axis 0 |
| Outputs differ from PyTorch | Model not in eval mode, dtype mismatch, unsupported behavior | Call `.eval()`, compare fp32 first, inspect operators |
| ONNX Runtime is slower | Small model, CPU overhead, provider mismatch | Compare providers, increase batch size, measure p95/p99 |
| Triton config fails | Config names or shapes do not match ONNX graph | Match `config.pbtxt` to graph inputs and outputs |

## 13. Production notes

For production, treat the ONNX file as a versioned artifact. Record:

- training checkpoint or commit used to export it,
- export script version,
- opset version,
- input and output names,
- expected dtypes and shapes,
- parity tolerance vs the source framework,
- latency benchmark environment,
- and runtime provider used for serving.

## 14. Exercises

1. Export the model with and without dynamic batch axes. Observe what changes.
2. Change `opset_version` and inspect the exported graph.
3. Compare PyTorch and ONNX Runtime at batch sizes 1, 8, 32, and 128.
4. Try dynamic quantization and measure whether it helps.
5. Build the Triton ONNX repository and compare it with the PyTorch repository in the next notebook.
6. Write a short note explaining when ONNX is helpful and when it adds unnecessary complexity.

## Summary

ONNX is a practical serving artifact. In this notebook you exported a PyTorch model, validated the graph, ran ONNX Runtime, compared output parity, benchmarked latency, tested dynamic batches, and prepared a Triton repository using the ONNX backend.

This is the missing bridge between local inference benchmarking and production model serving.